# Zensus 2022: 100 m grid cells for Mainz

This Colab-style notebook downloads official Zensus 2022 grid-cell data, extracts 100 m grid cells around Mainz, joins several demographic / housing variables, and visualizes the cells on an interactive map.

**Learning goals**

- Download open Zensus grid datasets from the official Destatis page.
- Work with German / European 100 m grid identifiers in EPSG:3035.
- Convert census grid IDs into geometries.
- Map selected socio-economic variables for a local study area.

**Data source**

- Official Zensus 2022 page: https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Bevoelkerung/Zensus2022/_inhalt.html
- The page states that GIS grid data are provided as OpenData ZIP archives. Each archive contains CSV files for 10 km, 1 km, and 100 m grids plus a dataset description.

Use the official dataset description files for exact column meanings and attribution wording.

In [ ]:
# Colab setup
!pip -q install geopandas shapely pyproj folium branca beautifulsoup4 requests tqdm mapclassify

In [ ]:
from pathlib import Path
from urllib.parse import urljoin
from zipfile import ZipFile
import io
import re

import branca.colormap as cm
import folium
import geopandas as gpd
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pyproj import Transformer
from shapely.geometry import box
from tqdm.auto import tqdm

## 1. Configuration

The notebook uses a bounding box around Mainz. This avoids dependence on geocoding services and keeps the example reproducible in Colab.

You can replace `MAINZ_BBOX_WGS84` with another study area. Format: `(min_lon, min_lat, max_lon, max_lat)`.

In [ ]:
ZENSUS_PAGE = "https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Bevoelkerung/Zensus2022/_inhalt.html"
CACHE_DIR = Path("/content/zensus_2022_grid")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Approximate Mainz bounding box in WGS84.
MAINZ_BBOX_WGS84 = (8.13, 49.90, 8.36, 50.10)


def bbox_wgs84_to_3035(bounds):
    min_lon, min_lat, max_lon, max_lat = bounds
    geom = gpd.GeoSeries([box(min_lon, min_lat, max_lon, max_lat)], crs="EPSG:4326").to_crs("EPSG:3035").iloc[0]
    return geom.bounds


mainz_bounds_3035 = bbox_wgs84_to_3035(MAINZ_BBOX_WGS84)

# Datasets to collect. The search terms are matched against link text on the official page.
ZENSUS_DATASETS = {
    "population": ["Bevölkerungszahlen", "Gitterzellen"],
    "avg_age": ["Durchschnittsalter", "Gitterzellen"],
    "foreign_share": ["Ausländeranteil", "Gitterzellen"],
    "avg_household_size": ["Durchschnittliche Haushaltsgröße", "Gitterzellen"],
    "avg_rent": ["Durchschnittliche Nettokaltmiete", "Gitterzellen"],
}

## 2. Find and download Zensus ZIP files

The Destatis page can change file URLs. Instead of hard-coding direct download URLs, this cell reads the official page and finds the current ZIP link by link text.

In [ ]:
def find_zensus_zip_links(page_url=ZENSUS_PAGE, dataset_terms=ZENSUS_DATASETS):
    response = requests.get(page_url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    for a in soup.find_all("a"):
        text = " ".join(a.get_text(" ", strip=True).split())
        href = a.get("href") or ""
        if not text or not href:
            continue
        links.append((text, urljoin(page_url, href)))

    result = {}
    for key, terms in dataset_terms.items():
        matches = []
        for text, url in links:
            text_lower = text.lower()
            url_lower = url.lower()
            if all(term.lower() in text_lower for term in terms):
                # Prefer actual ZIP downloads and skip dataset-description XLSX files.
                score = 0
                if ".zip" in url_lower or "download" in url_lower:
                    score += 2
                if "datensatzbeschreibung" in text_lower or url_lower.endswith(".xlsx"):
                    score -= 5
                matches.append((score, text, url))
        if not matches:
            raise ValueError(f"No link found for {key}: {terms}")
        matches = sorted(matches, key=lambda item: item[0], reverse=True)
        result[key] = {"link_text": matches[0][1], "url": matches[0][2]}
    return result

zip_links = find_zensus_zip_links()
pd.DataFrame(zip_links).T

In [ ]:
def download_file(url, out_path):
    out_path = Path(out_path)
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(out_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=out_path.name) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return out_path

zip_paths = {}
for key, item in zip_links.items():
    zip_paths[key] = download_file(item["url"], CACHE_DIR / f"{key}.zip")

zip_paths

## 3. Read the 100 m CSV tables

The official Destatis ZIP archives bundle several grid resolutions, usually 10 km, 1 km, and 100 m, plus a dataset description. The download therefore contains more than the 100 m file, but the notebook opens only the selected `100m-Gitter.csv` member.

To keep the Mainz example fast, the loader reads each 100 m CSV in chunks, keeps only the grid ID, midpoint coordinates, and the selected German value column, and filters to the Mainz bounding box before joining variables.

In [ ]:
def list_zip_csvs(zip_path):
    with ZipFile(zip_path) as zf:
        return [name for name in zf.namelist() if name.lower().endswith(".csv")]


def choose_100m_csv(zip_path):
    csvs = list_zip_csvs(zip_path)
    if not csvs:
        raise ValueError(f"No CSV in {zip_path}")
    ranked = sorted(
        csvs,
        key=lambda name: (
            "100m" in name.lower(),
            "100_m" in name.lower(),
            "100" in name.lower(),
        ),
        reverse=True,
    )
    return ranked[0]


for key, path in zip_paths.items():
    csvs = list_zip_csvs(path)
    selected = choose_100m_csv(path)
    print(f"{key}: selected {selected} ({len(csvs)} CSV files in ZIP)")

In [ ]:
# Official German column names in the selected Zensus 2022 100 m grid CSVs.
# Keeping them explicit is faster and safer than scanning all numeric columns.
ZENSUS_100M_COLUMNS = {
    "population": {"grid": "GITTER_ID_100m", "x": "x_mp_100m", "y": "y_mp_100m", "value": "Einwohner"},
    "avg_age": {"grid": "GITTER_ID_100m", "x": "x_mp_100m", "y": "y_mp_100m", "value": "Durchschnittsalter"},
    "foreign_share": {"grid": "GITTER_ID_100m", "x": "x_mp_100m", "y": "y_mp_100m", "value": "AnteilAuslaender"},
    "avg_household_size": {"grid": "GITTER_ID_100m", "x": "x_mp_100m", "y": "y_mp_100m", "value": "DurchschnHHGroesse"},
    "avg_rent": {"grid": "GITTER_ID_100m", "x": "x_mp_100m", "y": "y_mp_100m", "value": "durchschnMieteQM"},
}


def parse_numeric_column(series):
    s = series.astype(str).str.strip()
    s = s.replace({"": pd.NA, "-": pd.NA, "–": pd.NA, "—": pd.NA})
    s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def bbox_filter_from_midpoints(df, x_col, y_col, bbox_3035, pad=50):
    minx, miny, maxx, maxy = bbox_3035
    x = parse_numeric_column(df[x_col])
    y = parse_numeric_column(df[y_col])
    return x.between(minx - pad, maxx + pad) & y.between(miny - pad, maxy + pad)


def load_zensus_100m_value(zip_path, variable_name, bbox_3035=None, chunksize=250_000):
    member = choose_100m_csv(zip_path)
    cols = ZENSUS_100M_COLUMNS[variable_name]
    usecols = [cols["grid"], cols["x"], cols["y"], cols["value"]]
    parts = []
    rows_seen = 0

    with ZipFile(zip_path) as zf, zf.open(member) as f:
        reader = pd.read_csv(f, sep=";", dtype=str, usecols=usecols, chunksize=chunksize)
        for chunk in reader:
            rows_seen += len(chunk)
            if bbox_3035 is not None:
                chunk = chunk.loc[bbox_filter_from_midpoints(chunk, cols["x"], cols["y"], bbox_3035)]
            if chunk.empty:
                continue
            values = parse_numeric_column(chunk[cols["value"]])
            part = pd.DataFrame({
                "grid_id": chunk[cols["grid"]].astype(str),
                variable_name: values,
            }).dropna(subset=[variable_name])
            if not part.empty:
                parts.append(part)

    if parts:
        out = pd.concat(parts, ignore_index=True)
        out = out.groupby("grid_id", as_index=False)[variable_name].first()
    else:
        out = pd.DataFrame(columns=["grid_id", variable_name])

    scope = "bbox-filtered" if bbox_3035 is not None else "all"
    print(
        f"{variable_name}: {len(out):,} {scope} cells from {member}; "
        f"scanned {rows_seen:,} 100 m rows; value column={cols['value']!r}"
    )
    return out

value_tables = [load_zensus_100m_value(path, key, bbox_3035=mainz_bounds_3035) for key, path in zip_paths.items()]

## 4. Convert Zensus grid IDs to 100 m cell polygons

Zensus grid IDs typically encode ETRS89-LAEA coordinates (`EPSG:3035`) with northing and easting values. The parser below handles common INSPIRE-style patterns such as `CRS3035RES100mN...E...`.

In [ ]:
def parse_grid_id_bounds(grid_id, default_res=100):
    s = str(grid_id).upper().replace("_", "")

    # Common INSPIRE pattern: CRS3035RES100mN2688500E4213500
    m = re.search(r"RES(\d+)M?N(\d+)E(\d+)", s)
    if m:
        res = int(m.group(1))
        n = int(m.group(2))
        e = int(m.group(3))
        return e, n, e + res, n + res

    # Fallback: N2688500E4213500
    m = re.search(r"N(\d+)E(\d+)", s)
    if m:
        n = int(m.group(1))
        e = int(m.group(2))
        return e, n, e + default_res, n + default_res

    # Fallback: E4213500N2688500
    m = re.search(r"E(\d+)N(\d+)", s)
    if m:
        e = int(m.group(1))
        n = int(m.group(2))
        return e, n, e + default_res, n + default_res

    return None

mainz_bounds_3035

In [ ]:
# Join all variables by grid_id.
zensus = value_tables[0]
for table in value_tables[1:]:
    zensus = zensus.merge(table, on="grid_id", how="outer")

# Parse geometries and crop to Mainz.
bounds = zensus["grid_id"].map(parse_grid_id_bounds)
parsed = bounds.dropna()
geom_df = pd.DataFrame(parsed.tolist(), index=parsed.index, columns=["minx", "miny", "maxx", "maxy"])
zensus = zensus.loc[geom_df.index].join(geom_df)

minx, miny, maxx, maxy = mainz_bounds_3035
zensus_mainz = zensus[
    (zensus["maxx"] >= minx) & (zensus["minx"] <= maxx) &
    (zensus["maxy"] >= miny) & (zensus["miny"] <= maxy)
].copy()

zensus_mainz["geometry"] = [box(a, b, c, d) for a, b, c, d in zensus_mainz[["minx", "miny", "maxx", "maxy"]].to_numpy()]
gdf_mainz = gpd.GeoDataFrame(zensus_mainz, geometry="geometry", crs="EPSG:3035").to_crs("EPSG:4326")

print(f"Mainz subset: {len(gdf_mainz):,} 100 m cells")
gdf_mainz.head()

## 5. Interactive map

Change `VARIABLE_TO_MAP` to compare variables. Missing values are common because some official tables suppress values for privacy or statistical reliability.

In [ ]:
VARIABLE_TO_MAP = "population"  # try: avg_age, foreign_share, avg_household_size, avg_rent

plot_gdf = gdf_mainz.dropna(subset=[VARIABLE_TO_MAP]).copy()
center = plot_gdf.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

values = plot_gdf[VARIABLE_TO_MAP]
colormap = cm.linear.YlOrRd_09.scale(values.quantile(0.02), values.quantile(0.98))
colormap.caption = VARIABLE_TO_MAP


def style_function(feature):
    value = feature["properties"].get(VARIABLE_TO_MAP)
    if value is None:
        return {"fillOpacity": 0.05, "weight": 0.1, "color": "#888", "fillColor": "#cccccc"}
    return {
        "fillColor": colormap(value),
        "color": "#555555",
        "weight": 0.1,
        "fillOpacity": 0.65,
    }

folium.GeoJson(
    plot_gdf[["grid_id", VARIABLE_TO_MAP, "geometry"]],
    name=f"Zensus 100 m: {VARIABLE_TO_MAP}",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["grid_id", VARIABLE_TO_MAP], aliases=["Grid ID", VARIABLE_TO_MAP]),
).add_to(m)

folium.Rectangle(
    bounds=[[MAINZ_BBOX_WGS84[1], MAINZ_BBOX_WGS84[0]], [MAINZ_BBOX_WGS84[3], MAINZ_BBOX_WGS84[2]]],
    color="#1f77b4",
    weight=2,
    fill=False,
    tooltip="Mainz study bbox",
).add_to(m)

colormap.add_to(m)
folium.LayerControl().add_to(m)
m

## 6. Save the prepared Mainz grid

The output is written to Colab's temporary filesystem. Download it manually if needed. Do not commit large derived data files to the teaching repository unless licensing and file size have been checked.

In [ ]:
out_geojson = Path("/content/zensus_mainz_100m.geojson")
gdf_mainz.to_file(out_geojson, driver="GeoJSON")
out_geojson